# 🫵 Find the Right Troubleshooting Note - Solution

**An early retrieval exercise: index a small knowledge base, search it in several ways, and inspect what the results can actually support.**

The fictional **LumenDesk Developer Platform** has a compact support knowledge base. A developer describes a problem, and your job is to retrieve the note most likely to help. The collection deliberately contains related problems-authentication failures, retries, timeouts, webhook delivery, background jobs, and file processing - so merely finding a note about the same broad topic is not always enough.

This solution includes every assignment, the completed implementations, and short written responses in the exact answer locations.

**Prerequisite:** Demo 5a - ChromaDB Search and Keyword Extraction. This exercise reuses ChromaDB collections, semantic `query()`, metadata `where` filters, and document-text `where_document` filters. It does not require an LLM, an API key, Pandas, re-ranking, or similarity thresholds.

## Your mission


Complete five steps. Each step has one answer cell and produces a result that can be checked before you continue.

| Step | Deliverable |
|---|---|
| 1. Index | A Chroma collection containing all support notes and metadata |
| 2. Search semantically | A reusable semantic retriever and one evidence-based observation |
| 3. Constrain the search | A component-filtered retriever and one comparison observation |
| 4. Search literally | A document-text retriever and one comparison observation |
| 5. Improve and evaluate | A small routing policy, six-case evaluation, and a decision note |

**Completion criteria:** all 18 notes are indexed exactly once; result rows retain their IDs and metadata; component filters are respected; exact identifiers use literal matching; and conclusions distinguish relevant-looking text from evidence that actually addresses the query.


## Setup

Use the same Python environment as Demo 5. Run the installation cell only if Chroma is missing, then restart the kernel.

The first run may download Chroma's local MiniLM embedding model. All support notes are embedded in this notebook, so no external data service is used.


In [ ]:
# Uncomment only if Chroma is missing from your notebook environment.
# !pip install chromadb

In [ ]:
from html import escape
import re
from typing import Any, Callable, Final, TypedDict

import chromadb
from chromadb.api.models.Collection import Collection

from IPython.display import HTML, display

In [ ]:
class SupportNote(TypedDict):
    "A support record before it is added to Chroma."
    id: str
    title: str
    component: str
    kind: str
    error_code: str
    text: str


class SearchResult(TypedDict):
    "A normalized result returned by either Chroma query style."
    id: str
    document: str
    metadata: dict[str, Any]
    distance: float | None


class GoldenCase(TypedDict):
    "One expected retrieval behavior used in the final check."
    id: str
    query: str
    component: str | None
    expected_note_id: str | None

## Support knowledge base

The corpus contains 18 short, self-contained notes. Every note has:

- a stable ID used by our application;
- an exact error identifier that may appear in logs;
- a component and document kind stored as metadata;
- enough symptom, cause, and action text to stand on its own.

Several notes intentionally share words such as *timeout*, *retry*, *request*, and *signature*. That overlap makes the collection more representative than a set of unrelated documents.


In [ ]:
SUPPORT_NOTES: Final[list[SupportNote]] = [
    {
        "id": "auth-token-expired",
        "title": "Expired access token",
        "component": "auth",
        "kind": "troubleshooting",
        "error_code": "AUTH-1001",
        "text": (
            "AUTH-1001 - Access token expired. API requests return HTTP 401 with "
            "token_expired after a short-lived access token reaches its expiry time. "
            "Use the refresh token to obtain a new access token, then retry the request once."
        ),
    },
    {
        "id": "auth-key-rotation",
        "title": "Old API key used after rotation",
        "component": "auth",
        "kind": "troubleshooting",
        "error_code": "AUTH-1004",
        "text": (
            "AUTH-1004 - Invalid API key after credential rotation. Requests signed with the "
            "previous key return HTTP 401 invalid_api_key immediately after a key is rotated "
            "or revoked. Update the application's secret store and restart clients that cache credentials."
        ),
    },
    {
        "id": "auth-clock-skew",
        "title": "Signed request rejected because clocks differ",
        "component": "auth",
        "kind": "troubleshooting",
        "error_code": "AUTH-1012",
        "text": (
            "AUTH-1012 - Request signature expired. A signed request can be rejected when the "
            "client clock differs from the service clock by more than five minutes. Synchronize "
            "the client system clock and create a fresh signature before retrying."
        ),
    },
    {
        "id": "auth-insufficient-scope",
        "title": "Token lacks a required permission",
        "component": "auth",
        "kind": "reference",
        "error_code": "AUTH-1020",
        "text": (
            "AUTH-1020 - Insufficient scope. An otherwise valid token returns HTTP 403 when it "
            "does not include the permission required by the endpoint. Request a token with the "
            "documented scope; refreshing the same permissions will not help."
        ),
    },
    {
        "id": "rate-burst-limit",
        "title": "Short traffic burst exceeds the request limit",
        "component": "rate_limits",
        "kind": "troubleshooting",
        "error_code": "RATE-4290",
        "text": (
            "RATE-4290 - Burst request limit exceeded. A sudden spike can produce HTTP 429 even "
            "when the daily quota remains. Read Retry-After, pause for that interval, and spread "
            "future calls with exponential backoff and jitter."
        ),
    },
    {
        "id": "rate-daily-quota",
        "title": "Daily request quota exhausted",
        "component": "rate_limits",
        "kind": "reference",
        "error_code": "RATE-4292",
        "text": (
            "RATE-4292 - Daily quota exhausted. Every new API call returns HTTP 429 until the "
            "quota window resets. Waiting a few seconds does not restore capacity; reduce usage "
            "or request a larger daily allocation."
        ),
    },
    {
        "id": "rate-job-concurrency",
        "title": "Too many background jobs running",
        "component": "rate_limits",
        "kind": "reference",
        "error_code": "RATE-4294",
        "text": (
            "RATE-4294 - Concurrent job limit reached. Starting another export while the account "
            "already has the maximum number of running jobs returns HTTP 429. Wait for a running "
            "job to finish instead of resubmitting the same export repeatedly."
        ),
    },
    {
        "id": "webhook-signature",
        "title": "Webhook signature does not match",
        "component": "webhooks",
        "kind": "troubleshooting",
        "error_code": "HOOK-4012",
        "text": (
            "HOOK-4012 - Webhook signature mismatch. Verify the HMAC against the exact raw HTTP "
            "request body before parsing JSON or changing whitespace. Use the endpoint's current "
            "signing secret and compare the calculated value with the signature header."
        ),
    },
    {
        "id": "webhook-duplicates",
        "title": "The same webhook event is delivered more than once",
        "component": "webhooks",
        "kind": "concept",
        "error_code": "HOOK-2003",
        "text": (
            "HOOK-2003 - Duplicate webhook delivery. Acknowledgement can be lost, so the same "
            "event may reach an endpoint more than once. Store each event_id and make processing "
            "idempotent instead of assuming exactly-once delivery."
        ),
    },
    {
        "id": "webhook-retries",
        "title": "Webhook delivery is retried after a timeout",
        "component": "webhooks",
        "kind": "reference",
        "error_code": "HOOK-5031",
        "text": (
            "HOOK-5031 - Delivery endpoint unavailable. LumenDesk retries a webhook when the "
            "endpoint times out or returns a non-2xx response. Return a 2xx response quickly and "
            "move slow work to a queue; later attempts use exponential backoff."
        ),
    },
    {
        "id": "webhook-ordering",
        "title": "Related webhook events arrive out of order",
        "component": "webhooks",
        "kind": "concept",
        "error_code": "HOOK-2007",
        "text": (
            "HOOK-2007 - Webhook ordering is not guaranteed across deliveries. Compare each "
            "event's resource version or occurred_at value before updating local state; arrival "
            "time alone cannot determine which event is newer."
        ),
    },
    {
        "id": "webhook-replay",
        "title": "Replay a failed webhook delivery",
        "component": "webhooks",
        "kind": "how_to",
        "error_code": "HOOK-2040",
        "text": (
            "HOOK-2040 - Failed delivery replay. A failed webhook can be replayed from the delivery "
            "log for seven days. Select the original event and choose Replay; the replay keeps the "
            "same event_id, so the receiver must still be idempotent."
        ),
    },
    {
        "id": "job-status-polling",
        "title": "Poll a background export without overloading the API",
        "component": "jobs",
        "kind": "how_to",
        "error_code": "JOB-4080",
        "text": (
            "JOB-4080 - Background export still running. A create request returns a status URL; "
            "poll that URL no more than once every two seconds until the state is complete or failed. "
            "Creating the export again starts a second job rather than checking the first one."
        ),
    },
    {
        "id": "job-runtime-limit",
        "title": "Background job exceeded its maximum runtime",
        "component": "jobs",
        "kind": "troubleshooting",
        "error_code": "JOB-5040",
        "text": (
            "JOB-5040 - Job runtime limit exceeded. A large export can end in timed_out after the "
            "maximum processing window. Narrow the requested date range or split the export into "
            "smaller independent jobs before resubmitting."
        ),
    },
    {
        "id": "file-still-processing",
        "title": "Uploaded file is not ready to download",
        "component": "files",
        "kind": "troubleshooting",
        "error_code": "FILE-4091",
        "text": (
            "FILE-4091 - File is still processing. A download request returns HTTP 409 while an "
            "uploaded file has status=processing. Poll the file status endpoint and download only "
            "after the state changes to ready; uploading the same bytes again creates another file."
        ),
    },
    {
        "id": "file-checksum",
        "title": "Uploaded bytes fail checksum validation",
        "component": "files",
        "kind": "troubleshooting",
        "error_code": "FILE-4223",
        "text": (
            "FILE-4223 - Checksum mismatch. The checksum sent with an upload does not match the "
            "bytes received by the service. Recalculate the digest from the exact binary payload "
            "and make sure no text-mode conversion changes line endings."
        ),
    },
    {
        "id": "sdk-connect-timeout",
        "title": "Client cannot establish a connection in time",
        "component": "sdk",
        "kind": "troubleshooting",
        "error_code": "SDK-4081",
        "text": (
            "SDK-4081 - Connection timeout. The client could not establish a network connection "
            "before the configured connect timeout. Check DNS, proxy, and firewall settings before "
            "raising the timeout; no server response was received."
        ),
    },
    {
        "id": "sdk-safe-retries",
        "title": "Automatic retry of a non-idempotent request",
        "component": "sdk",
        "kind": "concept",
        "error_code": "SDK-4095",
        "text": (
            "SDK-4095 - Unsafe automatic retry. A connection can fail after the server accepted a "
            "request but before the client received the response. Retry GET safely; retry POST or "
            "PATCH only with an idempotency key or the operation may be applied twice."
        ),
    },
]

In [ ]:
def validate_notes(notes: list[SupportNote]) -> None:
    "Fail early if record IDs, error codes, or required metadata are malformed."
    required = {"id", "title", "component", "kind", "error_code", "text"}
    
    assert all(set(note) == required for note in notes)
    assert len({note["id"] for note in notes}) == len(notes)
    assert len({note["error_code"] for note in notes}) == len(notes)
    assert all(re.fullmatch(r"[A-Z]{3,5}-\d{4}", note["error_code"]) for note in notes)


def show_table(rows: list[dict[str, Any]]) -> None:
    "Display ordinary dictionaries as an escaped HTML table without Pandas."
    if not rows:
        print("No rows.")
        return
    
    columns = list(rows[0])
    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(
            f'<td style="text-align:left">{escape(str(row.get(column, '')))}</td>' for column in columns
        ) + "</tr>"
        for row in rows
    )
    
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

In [ ]:
validate_notes(SUPPORT_NOTES)

show_table([
    {
        "id": note["id"],
        "error_code": note["error_code"],
        "component": note["component"],
        "kind": note["kind"],
        "title": note["title"],
    }
    for note in SUPPORT_NOTES
])

In [ ]:
COLLECTION_NAME: Final[str] = "lumendesk-support-notes-v1"

client = chromadb.Client()

embedding_function = chromadb.utils.embedding_functions.ONNXMiniLM_L6_V2()
collection: Collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)

In [ ]:
def reset_collection(target: Collection) -> None:
    "Remove records from an earlier run so indexing remains repeatable."
    existing_ids: list[str] = target.get()["ids"]
    if existing_ids:
        target.delete(ids=existing_ids)

In [ ]:
reset_collection(collection)
print(f"Collection ready: {collection.count()} records")

## Step 1 - Build an inspectable index

**Purpose:** store the text and the fields needed for later retrieval decisions.

**Your assignment:** complete `index_notes(target, notes)` in the single answer cell below.

1. Add all notes in one Chroma call.
2. Use each note's stable `id` as its Chroma ID.
3. Index a document containing the title, error code, and note text.
4. Store `title`, `component`, `kind`, and `error_code` as metadata. Keep metadata values scalar.

**Expected result:** the collection contains exactly 18 records, and fetching `webhook-signature` returns the correct document and metadata.

**What this step demonstrates:** searchable text and application metadata have different jobs. Text contributes to similarity; metadata supports exact constraints and inspection.


### Solution


In [ ]:
# TODO 1 - completed in this solution: index every support note with useful metadata.
def index_notes(target: Collection, notes: list[SupportNote]) -> None:
    "Add all support notes to Chroma in one aligned batch."
    # The note text already contains its exact code; the title adds natural-language context.
    documents: list[str] = [
        f"{note['title']}\n{note['text']}" 
        for note in notes
    ]
    metadatas: list[dict[str, str]] = [
        {
            "title": note["title"],
            "component": note["component"],
            "kind": note["kind"],
            "error_code": note["error_code"],
        }
        for note in notes
    ]
    target.add(
        ids=[note["id"] for note in notes],
        documents=documents,
        metadatas=metadatas,
    )

In [ ]:
index_notes(collection, SUPPORT_NOTES)

In [ ]:
assert collection.count() == len(SUPPORT_NOTES) == 18

indexed_example: dict[str, Any] = collection.get(
    ids=["webhook-signature"],
    include=["documents", "metadatas"],
)

assert indexed_example["ids"] == ["webhook-signature"]
assert indexed_example["metadatas"][0]["component"] == "webhooks"
assert "HOOK-4012" in indexed_example["documents"][0]

show_table([{
    "id": indexed_example["ids"][0],
    **indexed_example["metadatas"][0],
    "document preview": indexed_example["documents"][0][:160] + "…",
}])

## Step 2 - Retrieve by meaning

**Purpose:** turn an ordinary symptom description into ranked support evidence.

**Your assignment:** complete `semantic_search(query, n_results)` and `STEP_2_OBSERVATION` in the single answer cell below.

1. Call `collection.query()` with one query string.
2. Request documents, metadata, and cosine distances for the nearest results. Chroma returns IDs automatically.
3. Return normalized `SearchResult` dictionaries through the supplied helper.
4. Run the three supplied queries, inspect full result rows, and complete the observation using evidence from your run.

**Expected result:** a paraphrased symptom can find a useful note even when the query and note do not use identical wording. The unsupported GraphQL question still receives neighbors.

**What this step demonstrates:** semantic similarity ranks nearby text; it does not prove that the collection contains an answer.


### Query-result helpers

Chroma returns one nested result list per query. We send one query at a time, so `flatten_query_results` reads index `0` and keeps the fields together. Lower cosine distance means a closer embedding neighbor.


In [ ]:
def flatten_query_results(raw: dict[str, Any]) -> list[SearchResult]:
    "Convert Chroma's one-query nested result into readable record dictionaries."
    return [
        SearchResult(
            id=record_id,
            document=document,
            metadata=dict(metadata),
            distance=float(distance),
        )
        for record_id, document, metadata, distance in zip(
            raw["ids"][0],
            raw["documents"][0],
            raw["metadatas"][0],
            raw["distances"][0],
        )
    ]


def result_rows(results: list[SearchResult], preview_chars: int = 150) -> list[dict[str, Any]]:
    "Prepare compact result rows while retaining the evidence-bearing identifiers."
    rows: list[dict[str, Any]] = []
    for rank, result in enumerate(results, start=1):
        distance = result["distance"]
        rows.append({
            "rank": rank,
            "distance": "" if distance is None else round(distance, 4),
            "id": result["id"],
            "component": result["metadata"]["component"],
            "error_code": result["metadata"]["error_code"],
            "title": result["metadata"]["title"],
            "preview": result["document"].replace("\n", " ")[:preview_chars] + "…",
        })
    
    return rows


def show_results(results: list[SearchResult]) -> None:
    "Display normalized retrieval results in ranked order."
    show_table(result_rows(results))

### Solution


In [ ]:
# TODO 2 - completed in this solution: implement semantic retrieval and record one observation.
def semantic_search(query: str, n_results: int = 3) -> list[SearchResult]:
    "Return the nearest support notes for an unrestricted semantic query."
    if not query.strip():
        raise ValueError("Query must not be empty.")
    raw: dict[str, Any] = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    
    return flatten_query_results(raw)

In [ ]:
PARAPHRASED_QUERY: Final[str] = "The same event reaches my endpoint more than once."
AMBIGUOUS_QUERY: Final[str] = "A request times out and then is tried again."
UNSUPPORTED_QUERY: Final[str] = "How do I subscribe to updates with GraphQL?"

In [ ]:
SEMANTIC_EXAMPLES: Final[list[tuple[str, str]]] = [
    ("Paraphrased symptom", PARAPHRASED_QUERY),
    ("Ambiguous symptom", AMBIGUOUS_QUERY),
    ("Unsupported question", UNSUPPORTED_QUERY),
]

for label, query in SEMANTIC_EXAMPLES:
    print(f"\n{label}: {query}")
    show_results(semantic_search(query))

**Observation**:

The duplicate-event query can retrieve the duplicate-delivery note through meaning, not an exact sentence match. The GraphQL question still gets ranked notes even though the corpus contains no GraphQL guidance; nearest-neighbor search always returns the closest available candidates when asked for results.

## Step 3 - Apply an explicit component constraint

**Purpose:** prevent semantically related notes from the wrong subsystem from entering the candidate set.

**Your assignment:** complete `semantic_search_in_component(query, component, n_results)` and `STEP_3_OBSERVATION` in the single answer cell below.

1. Validate that `component` is one of the values present in the corpus.
2. Keep the same semantic query used in Step 2.
3. Add a Chroma metadata `where` filter so only that component is eligible.
4. Compare unrestricted results with results constrained to `webhooks`.

**Expected result:** every filtered row has `component="webhooks"`; the result set focuses on webhook retry behavior rather than SDK, job, or rate-limit retry behavior.

**What this step demonstrates:** when the user or application knows a hard constraint, apply it before similarity ranking rather than asking embeddings to infer it.


### Solution


In [ ]:
# TODO 3 - completed in this solution: constrain semantic search by component.
def semantic_search_in_component(
    query: str,
    component: str,
    n_results: int = 3,
) -> list[SearchResult]:
    "Return semantic neighbors selected only from one support component."
    known_components: set[str] = {note["component"] for note in SUPPORT_NOTES}

    if component not in known_components:
        raise ValueError(f"Unknown component {component!r}. Choose from {sorted(known_components)}")

    raw: dict[str, Any] = collection.query(
        query_texts=[query],
        n_results=n_results,
        where={"component": component},
        include=["documents", "metadatas", "distances"],
    )
    
    return flatten_query_results(raw)

In [ ]:
RETRY_QUERY: Final[str] = AMBIGUOUS_QUERY

print("UNRESTRICTED")
unrestricted_retry_results: list[SearchResult] = semantic_search(RETRY_QUERY, n_results=5)
show_results(unrestricted_retry_results)

print("\nWEBHOOKS ONLY")
webhook_retry_results: list[SearchResult] = semantic_search_in_component(
    RETRY_QUERY,
    component="webhooks",
    n_results=5,
)

show_results(webhook_retry_results)

assert webhook_retry_results
assert all(result["metadata"]["component"] == "webhooks" for result in webhook_retry_results)

**Observation**:

The unrestricted query can mix several legitimate meanings of retry and timeout. The metadata filter does not improve the wording of the query; it removes records from other components before Chroma ranks the remaining webhook candidates.

## Step 4 - Compare literal and semantic matching

**Purpose:** choose a search method that fits the information present in the query.

**Your assignment:** complete `literal_search(needle, component)` and `STEP_4_OBSERVATION` in the single answer cell below.

1. Use `collection.get()` with `where_document={"$contains": needle}`.
2. Add a metadata `where` filter only when a component is supplied.
3. Normalize the flat `get()` response with the supplied helper.
4. Compare an exact error code, a paraphrased symptom, and an ambiguous retry description using the supplied cells.

**Expected result:** literal search reliably locates the exact error code; an entire paraphrased sentence has no literal match; semantic search still supplies ranked candidates.

**What this step demonstrates:** literal and semantic search solve different retrieval problems. Exact identifiers reward character-level matching; paraphrases reward meaning-level matching.

### Full-text-result helper

Unlike `query()`, Chroma's `get()` response is not nested by query and does not contain distances. We keep `distance=None` so both search styles can share one display format.

In [ ]:
def flatten_get_results(raw: dict[str, Any]) -> list[SearchResult]:
    "Normalize Chroma's flat get result without inventing a similarity distance."
    return [
        SearchResult(
            id=record_id,
            document=document,
            metadata=dict(metadata),
            distance=None,
        )
        for record_id, document, metadata in zip(
            raw["ids"],
            raw["documents"],
            raw["metadatas"],
        )
    ]

### Solution


In [ ]:
# TODO 4 — completed in this solution: implement literal document-text search.
def literal_search(needle: str, component: str | None = None) -> list[SearchResult]:
    "Return records whose stored document contains an exact case-sensitive string."
    if not needle:
        raise ValueError("Search text must not be empty.")
    arguments: dict[str, Any] = {
        "where_document": {"$contains": needle},
        "include": ["documents", "metadatas"],
    }
    if component is not None:
        arguments["where"] = {"component": component}
    raw: dict[str, Any] = collection.get(**arguments)
    
    return flatten_get_results(raw)

In [ ]:
EXACT_CODE: Final[str] = "HOOK-4012"

print(f"LITERAL SEARCH - exact identifier: {EXACT_CODE}")
show_results(literal_search(EXACT_CODE))

print(f"\nSEMANTIC SEARCH - exact identifier: {EXACT_CODE}")
show_results(semantic_search(EXACT_CODE))

print(f"\nLITERAL SEARCH - full paraphrased sentence: {PARAPHRASED_QUERY}")
paraphrase_literal_results: list[SearchResult] = literal_search(PARAPHRASED_QUERY)
show_results(paraphrase_literal_results)

print(f"\nSEMANTIC SEARCH - paraphrased symptom: {PARAPHRASED_QUERY}")
show_results(semantic_search(PARAPHRASED_QUERY))

print("\nLITERAL SEARCH - ambiguous term: timeout")
show_results(literal_search("timeout"))

print(f"\nSEMANTIC SEARCH - complete ambiguous question: {AMBIGUOUS_QUERY}")
show_results(semantic_search(AMBIGUOUS_QUERY))

assert [result["id"] for result in literal_search(EXACT_CODE)] == ["webhook-signature"]
assert paraphrase_literal_results == []

**Observation**:

Literal search is the stronger choice for the opaque identifier HOOK-4012 because the characters themselves are the evidence. It is the wrong choice for a user's paraphrased symptom: requiring that whole sentence to occur verbatim returns no rows, while semantic search can still find a note about duplicate delivery. Searching literally for the ambiguous word timeout returns un-ranked occurrences; semantic search uses the rest of the question to rank different meanings of timeout.

## Step 5 - Combine the signals and evaluate

**Purpose:** turn the earlier observations into a small, explicit retrieval policy and check it across several cases.

**Your assignment:** complete `search_support_notes(query, component, n_results)` and `STEP_5_DECISION` in the single answer cell below.

Use this policy:

1. If the query contains an error identifier, use literal search for that identifier.
2. Otherwise, if the application supplies a component, use component-filtered semantic search.
3. Otherwise, use unrestricted semantic search.

Run the supplied six-case evaluation and compare the baseline with your policy. In the decision note, describe one gain and one limitation.

**Expected result:** exact-code routing returns the matching record; filtered queries remain component-pure; answerable cases show whether the expected note appears in the first three results; and the unsupported case is inspected without pretending that a returned neighbor is an answer.

**What this step demonstrates:** a retriever can combine simple signals transparently. A small golden set catches behavior across query types, but retrieval evaluation alone does not establish answer correctness or a safe refusal threshold.

### Golden cases

Five cases have a known useful note. The final query is deliberately unsupported; its expected ID is `None`, so it is displayed but excluded from hit calculations.

The `component` value represents an explicit UI selection or known route - not a component secretly inferred from the answer key.

In [ ]:
GOLDEN_CASES: Final[list[GoldenCase]] = [
    {
        "id": "G-01",
        "query": "Logs show HOOK-4012 when my endpoint receives an event.",
        "component": None,
        "expected_note_id": "webhook-signature",
    },
    {
        "id": "G-02",
        "query": "Our handler sometimes processes the same event twice.",
        "component": "webhooks",
        "expected_note_id": "webhook-duplicates",
    },
    {
        "id": "G-03",
        "query": "A write may have reached the server, but the client lost the response. Can it retry?",
        "component": "sdk",
        "expected_note_id": "sdk-safe-retries",
    },
    {
        "id": "G-04",
        "query": "The upload finished, but downloading it immediately returns a conflict.",
        "component": "files",
        "expected_note_id": "file-still-processing",
    },
    {
        "id": "G-05",
        "query": "Signed requests suddenly say the signature is expired on one machine.",
        "component": "auth",
        "expected_note_id": "auth-clock-skew",
    },
    {
        "id": "G-06",
        "query": "How can I subscribe to record updates with GraphQL?",
        "component": None,
        "expected_note_id": None,
    },
]

In [ ]:
show_table([dict(case) for case in GOLDEN_CASES])

### Identifier helper

In [ ]:
ERROR_CODE_PATTERN: Final[re.Pattern[str]] = re.compile(r"\b[A-Z]{3,5}-\d{4}\b")


def extract_error_code(query: str) -> str | None:
    "Return the first supported error-code shape in the query, if present."
    match = ERROR_CODE_PATTERN.search(query.upper())
    return match.group(0) if match else None

### Solution

In [ ]:
# TODO 5 - completed in this solution: route by explicit evidence and justify the policy.
def search_support_notes(
    query: str,
    component: str | None = None,
    n_results: int = 3,
) -> list[SearchResult]:
    "Choose literal or semantic retrieval from observable query constraints."
    error_code = extract_error_code(query)
    if error_code is not None:
        return literal_search(error_code, component=component)[:n_results]
    if component is not None:
        return semantic_search_in_component(query, component, n_results=n_results)
    
    return semantic_search(query, n_results=n_results)

### Evaluation helpers - supplied

For answerable cases, `rank` is the one-based position of the expected note and `hit_at_3` reports whether it appears in the first three rows. For the unsupported case, both values remain blank because there is no correct note to retrieve.

In [ ]:
SearchFunction = Callable[[str, str | None, int], list[SearchResult]]


def baseline_search(
    query: str,
    component: str | None = None,
    n_results: int = 3,
) -> list[SearchResult]:
    "Provide an unrestricted semantic baseline; component is intentionally ignored."
    del component
    return semantic_search(query, n_results=n_results)


def evaluate_search(
    retriever: SearchFunction,
    cases: list[GoldenCase],
    n_results: int = 3,
) -> list[dict[str, Any]]:
    "Evaluate expected-note rank while retaining unsupported cases for inspection."
    rows: list[dict[str, Any]] = []
    for case in cases:
        results = retriever(case["query"], case["component"], n_results)
        result_ids = [result["id"] for result in results]
        expected_id = case["expected_note_id"]
        rank = result_ids.index(expected_id) + 1 if expected_id in result_ids else None
        rows.append({
            "case": case["id"],
            "component": case["component"] or "—",
            "expected": expected_id or "unsupported",
            "rank": rank,
            "hit_at_3": None if expected_id is None else rank is not None and rank <= 3,
            "top_ids": result_ids,
        })
    return rows


def answerable_recall_at_3(rows: list[dict[str, Any]]) -> float:
    "Return the fraction of answerable cases whose expected note is in the top three."
    answerable = [row for row in rows if row["hit_at_3"] is not None]
    return sum(bool(row["hit_at_3"]) for row in answerable) / len(answerable)

In [ ]:
baseline_rows: list[dict[str, Any]] = evaluate_search(baseline_search, GOLDEN_CASES)
policy_rows: list[dict[str, Any]] = evaluate_search(search_support_notes, GOLDEN_CASES)

print("UNRESTRICTED SEMANTIC BASELINE")
show_table(baseline_rows)
print(f"Answerable recall@3: {answerable_recall_at_3(baseline_rows):.2f}")

print("\nEXPLICIT ROUTING POLICY")
show_table(policy_rows)
print(f"Answerable recall@3: {answerable_recall_at_3(policy_rows):.2f}")

# These checks validate deterministic policy behavior, not model-dependent ranking.
assert search_support_notes("Investigate HOOK-4012")[0]["id"] == "webhook-signature"

for case in GOLDEN_CASES:
    if case["component"] is not None:
        results = search_support_notes(case["query"], case["component"])
        assert all(result["metadata"]["component"] == case["component"] for result in results)

**Decision note**:

The policy gives exact identifiers deterministic treatment and prevents explicit component constraints from being ignored. Its limitation is unchanged from Step 2: an unsupported question still receives nearest neighbors. Without a taught and tested abstention rule, those neighbors must be inspected rather than treated as an answer.

## Inspect the unsupported case

The knowledge base contains no GraphQL subscription guidance. Nevertheless, semantic retrieval returns the closest available notes. Read them and decide whether any actually supports the requested capability.

At this point in the course, the correct conclusion is an observation about missing evidence—not an invented similarity threshold and not an automated refusal mechanism.

In [ ]:
unsupported_case: GoldenCase = GOLDEN_CASES[-1]

unsupported_results: list[SearchResult] = search_support_notes(
    unsupported_case["query"],
    unsupported_case["component"],
)

show_results(unsupported_results)

## Takeaways

- Store stable IDs and useful metadata so retrieved evidence can be inspected and constrained.
- Semantic retrieval is valuable for paraphrases, but a nearest neighbor is not automatically an answer.
- Metadata filtering is appropriate for explicit constraints such as component, product, tenant, language, or document type.
- Literal document search is dependable for exact identifiers; it is brittle for ordinary paraphrased questions.
- A small, transparent routing policy can combine those strengths without introducing reranking or an LLM.
- Retrieval metrics measure whether expected evidence was found. They do not establish that a generated answer is faithful, complete, or safe to provide.

## References

- [Chroma collection query and get API](https://docs.trychroma.com/reference/python/collection)
- [Chroma metadata filtering](https://docs.trychroma.com/docs/querying-collections/metadata-filtering)
- [Chroma full-text search](https://docs.trychroma.com/docs/querying-collections/full-text-search)

<hr/>
